# 🏠 03 — Regression on California Housing Dataset

Compares **Linear Regression**, **Ridge**, **Lasso**, and **Random Forest Regressor**.

Uses the reusable `evaluator.compare_models()` and `visualizer.plot_actual_vs_predicted()` utilities.

In [ ]:
import sys
sys.path.insert(0, 'src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

from evaluator import compare_models, evaluate_regressor
from visualizer import plot_actual_vs_predicted, plot_feature_importance

housing = fetch_california_housing(as_frame=True)
df = housing.frame
print('Dataset shape:', df.shape)
df.head()

In [ ]:
# --- EDA ---
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, col in zip(axes.flat, df.columns):
    ax.hist(df[col], bins=40, color='steelblue', edgecolor='white')
    ax.set_title(col, fontsize=9)
plt.suptitle('Feature Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Prepare data ---
X = df.drop(columns=['MedHouseVal']).values
y = df['MedHouseVal'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
print(f'Train: {X_train_s.shape}  |  Test: {X_test_s.shape}')

In [ ]:
# --- Compare models ---
models = {
    'Linear Regression':   LinearRegression(),
    'Ridge (alpha=1)':     Ridge(alpha=1.0),
    'Lasso (alpha=0.01)':  Lasso(alpha=0.01, max_iter=5000),
    'Random Forest':       RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
}

results_df = compare_models(models, X_train_s, X_test_s, y_train, y_test, task='regression')
print('\n=== Model Comparison ===')
results_df

In [ ]:
# --- Best model deep-dive: Random Forest ---
best = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
best.fit(X_train_s, y_train)
y_pred = best.predict(X_test_s)

evaluate_regressor(best, X_test_s, y_test)

plot_actual_vs_predicted(y_test, y_pred, title='Random Forest: Actual vs Predicted')
plot_feature_importance(best.feature_importances_, housing.feature_names,
                        title='Random Forest Feature Importances')